In [1]:
import sys
sys.path.insert(0,'C:\\Users\\[LEARNER_NAME_OMITTED]\\.v\\hr\\20261019T063318-5946e910')
import run_recovery as _kernel_management
_kernel_management.write_kernel_runtime_identity('C:\\Users\\[LEARNER_NAME_OMITTED]\\.v\\hr\\20261019T063318-5946e910',39612)
import recovery_bridge as recovery
recovery.install('C:\\Users\\[LEARNER_NAME_OMITTED]\\.v\\hr\\20261019T063318-5946e910\\source\\HaiLong_Colab_Exercises','C:\\Users\\[LEARNER_NAME_OMITTED]\\.v\\hr\\20261019T063318-5946e910')


In [2]:
# Colab setup. Run this cell before the exercise cells.
from pathlib import Path
import os, sys, zipfile

USE_GOOGLE_DRIVE = True  #@param {type:"boolean"}
QUICK_RUN = False  #@param {type:"boolean"}
ARCHIVE_PATH = "/content/drive/MyDrive/HaiLong_Colab_Exercises_2026-10-03.zip"  #@param {type:"string"}

try:
    from google.colab import drive, files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if USE_GOOGLE_DRIVE:
        if not Path('/content/drive/MyDrive').is_dir():
            drive.mount('/content/drive', timeout_ms=300000)
    COURSE_ROOT = Path('/content/HaiLong_Colab_Exercises')
    if not (COURSE_ROOT / 'colab_runtime.py').is_file():
        archive = Path(ARCHIVE_PATH)
        if not archive.is_file():
            print('Select HaiLong_Colab_Exercises_2026-10-03.zip from your computer.')
            uploaded = files.upload()
            choices = [Path(n) for n in uploaded if n.endswith('.zip')]
            if len(choices) != 1:
                raise ValueError('Upload exactly one course ZIP file.')
            archive = choices[0]
        with zipfile.ZipFile(archive) as z:
            base = Path('/content').resolve()
            for member in z.infolist():
                target = (base / member.filename).resolve()
                if not target.is_relative_to(base) or not member.filename.startswith('HaiLong_Colab_Exercises/'):
                    raise ValueError('This is not the expected course archive.')
            z.extractall(base)
    if not (COURSE_ROOT / 'colab_runtime.py').is_file():
        raise FileNotFoundError('The ZIP must contain HaiLong_Colab_Exercises/colab_runtime.py.')
else:
    # Also permits local checks without assuming the current notebook directory.
    COURSE_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                        if (p / 'colab_runtime.py').is_file()), None)
    if COURSE_ROOT is None:
        raise RuntimeError('Open this notebook in Colab or run it inside the extracted course folder.')

os.environ['COURSE_SMOKE'] = '1' if QUICK_RUN else '0'
os.environ['WANDB_MODE'] = 'offline'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
sys.path.insert(0, str(COURSE_ROOT))
from colab_runtime import setup_course, finish_run
# Filled individually for each notebook by the builder.

COLAB_CONTEXT = setup_course(COURSE_ROOT, '03_fastai', 'book', '15_collaborative_filtering.ipynb', USE_GOOGLE_DRIVE, install=IN_COLAB)

Exercise folder: C:\Users\[LOCAL_USER]\.v\hr\20261019T063318-5946e910\source\HaiLong_Colab_Exercises\03_fastai
Run mode: FULL EXERCISE
Results: C:\Users\[LOCAL_USER]\.v\hr\20261019T063318-5946e910\source\HaiLong_Colab_Exercises\03_fastai\results


RESUMED EVIDENCE: ten verified prior five-epoch fits reused, zero of those fits retrained.


In [3]:
import fastbook  # Installed by the Colab setup cell.

In [4]:
#hide
from fastbook import *

In [5]:
from fastai.collab import *
from fastai.tabular.all import *
path = untar_data(URLs.ML_100k)

In [6]:
ratings = pd.read_csv(path/'u.data', delimiter='\t', header=None,
                      names=['user','movie','rating','timestamp'])
ratings.head()

,user,movie,rating,timestamp
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116
3,244,51,2,880606923
4,166,346,1,886397596


In [7]:
movies = pd.read_csv(path/'u.item',  delimiter='|', encoding='latin-1',
                     usecols=(0,1), names=('movie','title'), header=None)
movies.head()

,movie,title
0,1,Toy Story (1995)
1,2,GoldenEye (1995)
2,3,Four Rooms (1995)
3,4,Get Shorty (1995)
4,5,Copycat (1995)


In [8]:
ratings = ratings.merge(movies)
ratings.head()
if QUICK_RUN: ratings = ratings.sample(n=min(5000,len(ratings)), random_state=42).reset_index(drop=True)

In [9]:
# Objective questionnaire Q5 and Q7: crosstab and a pure-Python dot product.
import json, time, hashlib, gc
from pathlib import Path
from torch.utils.data import Dataset,DataLoader
from colab_runtime import download,safe_extract
rating_crosstab=pd.crosstab(ratings['user'],ratings['title'],values=ratings['rating'],aggfunc='mean')
display(rating_crosstab.iloc[:8,:8])
Path('results').mkdir(exist_ok=True)
rating_crosstab.iloc[:20,:20].to_csv('results/movielens_100k_crosstab_preview.csv')
left=[.9,.8,-.6];right=[.98,.9,-.9]
python_dot_product=sum(a*b for a,b in zip(left,right))
assert abs(python_dot_product-2.142)<1e-12
print('Pure-Python dot product:',python_dot_product)
del rating_crosstab

# Full-data extension: use every rating in the stable public ML-25M release.
# QUICK_RUN is explicitly a check of the original 100k data, not this homework.
full_path=Path('data/ml-25m')
if QUICK_RUN:
    full_ratings=ratings[['user','movie','rating']].copy()
    full_movies=movies.copy();dataset_label='ML-100k QUICK CHECK'
else:
    if not (full_path/'ratings.csv').is_file():
        archive=download('https://files.grouplens.org/datasets/movielens/ml-25m.zip','data/ml-25m.zip')
        safe_extract(archive,Path('data'))
    full_ratings=pd.read_csv(full_path/'ratings.csv',usecols=['userId','movieId','rating'],
                             dtype={'userId':'int32','movieId':'int32','rating':'float32'}).rename(columns={'userId':'user','movieId':'movie'})
    full_movies=pd.read_csv(full_path/'movies.csv').rename(columns={'movieId':'movie'})
    assert len(full_ratings)==25000095,'All ML-25M ratings are required, without downsampling.'
    dataset_label='ML-25M all 25,000,095 ratings'

# Tensor-backed datasets avoid materializing a 162k-by-62k dense user/movie table.
user_codes,user_ids=pd.factorize(full_ratings['user'],sort=True)
movie_codes,movie_ids=pd.factorize(full_ratings['movie'],sort=True)
full_x=torch.from_numpy(np.column_stack((user_codes,movie_codes))).long()
full_y=torch.from_numpy(full_ratings['rating'].to_numpy(copy=True)).float()
levels=torch.tensor(sorted(full_ratings['rating'].unique()),dtype=torch.float32)
class_y=torch.searchsorted(levels,full_y).long()
assert torch.allclose(levels[class_y],full_y)
assert QUICK_RUN or len(levels)==10 # Half-star ratings 0.5,...,5.0 need ten classes.
permutation=np.random.default_rng(42).permutation(len(full_y))
valid_indices=permutation[:len(permutation)//5];train_indices=permutation[len(permutation)//5:]
split_sha=hashlib.sha256(valid_indices.tobytes()).hexdigest()
np.save('results/movielens_full_valid_indices.npy',valid_indices)
np.savez('results/movielens_full_id_maps.npz',user_ids=np.asarray(user_ids),movie_ids=np.asarray(movie_ids),rating_levels=levels.numpy())
class RatingRows(Dataset):
    def __init__(self,indices,classification=False):self.indices,self.classification=indices,classification
    def __len__(self):return len(self.indices)
    def __getitem__(self,i):
        index=int(self.indices[i]);return full_x[index],(class_y if self.classification else full_y)[index]

FULL_BATCH_SIZE=1024
def full_rating_loaders(classification=False):
    generator=torch.Generator().manual_seed(42)
    train_dl=DataLoader(RatingRows(train_indices,classification),batch_size=FULL_BATCH_SIZE,shuffle=True,num_workers=0,generator=generator)
    valid_dl=DataLoader(RatingRows(valid_indices,classification),batch_size=FULL_BATCH_SIZE,shuffle=False,num_workers=0)
    return DataLoaders(train_dl,valid_dl,device=torch.device('cuda' if torch.cuda.is_available() else 'cpu'))

# Extra movie columns are static genre features; no held-out ratings become inputs.
movie_table=full_movies.set_index('movie').reindex(movie_ids)
genre_frame=movie_table['genres'].fillna('').str.get_dummies('|') if 'genres' in movie_table else pd.DataFrame(index=movie_table.index)
genre_features=torch.tensor(genre_frame.to_numpy(),dtype=torch.float32)
class FullMovieLensNN(nn.Module):
    def __init__(self,classification=False,use_genres=False):
        super().__init__();self.classification=classification;self.use_genres=use_genres
        self.users=Embedding(len(user_ids),50);self.movies=Embedding(len(movie_ids),50)
        self.register_buffer('genre_features',genre_features)
        width=100+(genre_features.shape[1] if use_genres else 0)
        self.layers=nn.Sequential(nn.Linear(width,100),nn.ReLU(),nn.Linear(100,len(levels) if classification else 1))
    def forward(self,x):
        parts=[self.users(x[:,0]),self.movies(x[:,1])]
        if self.use_genres:parts.append(self.genre_features[x[:,1]])
        outputs=self.layers(torch.cat(parts,dim=1))
        return outputs if self.classification else torch.sigmoid(outputs)*5.5

@torch.no_grad()
def full_rating_evaluation(learner,classification):
    learner.model.eval();device=next(learner.model.parameters()).device
    squared=absolute=correct=rows=0;confusion=torch.zeros(len(levels),len(levels),dtype=torch.int64)
    for cats,targets in learner.dls.valid:
        outputs=learner.model(cats.to(device)).cpu()
        if classification:
            probabilities=outputs.softmax(-1);predictions=probabilities@levels
            expected_classes=targets.long().reshape(-1);truth=levels[expected_classes];chosen=outputs.argmax(-1)
        else:
            predictions=outputs.reshape(-1);truth=targets.float().reshape(-1)
            expected_classes=torch.searchsorted(levels,truth)
            chosen=(predictions[:,None]-levels[None,:]).abs().argmin(-1)
        errors=predictions-truth;squared+=float(errors.square().sum());absolute+=float(errors.abs().sum())
        correct+=int((chosen==expected_classes).sum());rows+=len(truth)
        confusion+=torch.bincount(expected_classes*len(levels)+chosen,minlength=len(levels)**2).reshape(len(levels),len(levels))
    assert rows==len(valid_indices)
    return {'rmse':(squared/rows)**.5,'mae':absolute/rows,'rating_class_accuracy':correct/rows,'validation_rows':rows},confusion

experiments=[('nn_mse',False,False),('nn_cross_entropy',True,False),('nn_mse_with_genres',False,True)]
comparison=[];full_epochs=1 if QUICK_RUN else 5


title,'Til There Was You (1997),1-900 (1994),101 Dalmatians (1996),12 Angry Men (1957),187 (1997),2 Days in the Valley (1996),"20,000 Leagues Under the Sea (1954)",2001: A Space Odyssey (1968)
user,,,,,,,,
1,NaN,NaN,2.0,5.0,NaN,NaN,3.0,4.0
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,2.0,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,2.0,NaN,NaN,NaN,NaN,4.0
6,NaN,NaN,NaN,4.0,NaN,NaN,NaN,5.0
7,NaN,NaN,NaN,4.0,NaN,NaN,5.0,5.0
8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Pure-Python dot product: 2.1420000000000003


In [10]:
recovery.validate_reconstructed_data(globals())
comparison = recovery.prior_configurations()
recovery.attach_resume(globals())
name, classification, use_genres = experiments[2]


In [11]:
set_seed(42);experiment_dls=full_rating_loaders(classification)
model=FullMovieLensNN(classification,use_genres)
objective=CrossEntropyLossFlat() if classification else MSELossFlat()
experiment=Learner(experiment_dls,model,loss_func=objective)
start=time.perf_counter();experiment.fit_one_cycle(full_epochs,5e-3,wd=.01)
metrics,confusion=full_rating_evaluation(experiment,classification)
row={'model':name,'dataset':dataset_label,'source_rows':len(full_y),'training_rows':len(train_indices),
     'validation_split_sha256':split_sha,'epochs':full_epochs,'batch_size':FULL_BATCH_SIZE,
     'class_levels':levels.tolist(),'objective':'cross_entropy' if classification else 'mse',
     'genres_used':bool(use_genres and genre_features.shape[1]),'genre_columns':genre_frame.columns.tolist(),
     'parameters':sum(p.numel() for p in model.parameters()),
     'training_seconds':time.perf_counter()-start,'quick_run':QUICK_RUN,**metrics}
assert np.isfinite(row['rmse']) and np.isfinite(row['mae'])
comparison.append(row)
torch.save({'state_dict':model.state_dict(),'architecture':repr(model),'configuration':row},f'models/{name}_full_movielens.pt')
pd.DataFrame(confusion.numpy(),index=levels.tolist(),columns=levels.tolist()).to_csv(f'results/{name}_rating_confusion.csv')
pd.DataFrame(experiment.recorder.values,columns=['train_loss','valid_loss']).to_csv(f'results/{name}_training_history.csv',index=False)
pd.DataFrame(comparison).to_csv('results/full_movielens_model_comparison.csv',index=False)
del experiment,model,experiment_dls;gc.collect()
if torch.cuda.is_available():torch.cuda.empty_cache()


Storage check (before_fit): artifacts 0.243 GB; configured budget 12.000 GB; additional estimate 0.580 GB; runtime free 621.585 GB. Budget is not Drive quota.
Explicit checkpoint recovery: next epoch 3 of 5
Storage check (checkpoint_write): artifacts 0.243 GB; configured budget 12.000 GB; additional estimate 0.191 GB; runtime free 621.585 GB. Budget is not Drive quota.


epoch,train_loss,valid_loss,time
3,0.566559,0.604538,05:02
4,0.506704,0.606413,05:10


Storage check (checkpoint_write): artifacts 0.425 GB; configured budget 12.000 GB; additional estimate 0.191 GB; runtime free 617.159 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.607 GB; configured budget 12.000 GB; additional estimate 0.191 GB; runtime free 613.359 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.607 GB; configured budget 12.000 GB; additional estimate 0.057 GB; runtime free 613.359 GB. Budget is not Drive quota.


In [12]:
recovery.annotate_comparison(globals())


In [13]:

# Online context: same release, different split/model/tuning. No leaderboard claim.
benchmark={'source':'https://codex.lenskit.org/movielens/ML25M/als-biased.html',
 'split_source':'https://codex.lenskit.org/movielens/ML25M/splitting.html',
 'reference_rmse':.7918119847363368,'reference_model':'LensKit ALS biased MF, 471 factors, tuned parameters',
 'reference_protocol':'global temporal train/tune/test windows; test users restricted to those seen in training',
 'our_protocol':'seed42 random row 80/20 validation split, no separate test/tuning window',
 'directly_comparable':False,'source_status':'work in progress; illustrative published online result, not a validated target',
 'differences_from_reference':{r['model']:r['rmse']-.7918119847363368 for r in comparison},
 'genre_improvement_rmse':comparison[0]['rmse']-comparison[2]['rmse'],
 'cross_entropy_minus_mse_rmse':comparison[1]['rmse']-comparison[0]['rmse']}
(Path('results')/'full_movielens_benchmark_context.json').write_text(json.dumps(benchmark,indent=2))
(Path('results')/'full_movielens_run_evidence.json').write_text(json.dumps({'models':comparison,'benchmark':benchmark,
 'dataset_source':'https://grouplens.org/datasets/movielens/25m/',
 'dataset_terms':'https://files.grouplens.org/datasets/movielens/ml-25m-README.html',
 'questionnaire_code_mapping':{'Q5':'crosstab above','Q7':'pure-Python dot product above','Q13':'original Example class','Q15':'original DotProduct training cells'},
 'provider_course_completion_claimed':False},indent=2))
display(pd.DataFrame(comparison));print(json.dumps(benchmark,indent=2))
print('Cross-entropy predicts ten half-star classes on ML-25M; its probability-weighted expected rating allows an RMSE/MAE comparison with regression.')
print('Genre features improve RMSE only if the measured difference above is positive. Online benchmark protocols differ; the difference is descriptive, not a ranking.')

,model,dataset,source_rows,training_rows,validation_split_sha256,epochs,batch_size,class_levels,objective,genres_used,genre_columns,parameters,training_seconds,quick_run,rmse,mae,rating_class_accuracy,validation_rows,training_seconds_scope,original_completed_epochs_before_recovery,actual_new_epochs_trained,original_checkpoint_prefix_elapsed_seconds,checkpoint_sha256
0,nn_mse,"ML-25M all 25,000,095 ratings",25000095,20000076,4fbcebdec690e07e15fbd5d5f0016a309ff5c2da8fb516aaae47c896568395ce,5,1024,"[0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0]",mse,False,"[(no genres listed), Action, Adventure, Animation, Children, Comedy, Crime, Documentary, Drama, Fantasy, Film-Noir, Horror, IMAX, Musical, Mystery, Romance, Sci-Fi, Thriller, War, Western]",11089601,1435.386894,False,0.787495,0.592119,0.294155,5000019,unchanged prior full fit plus original evaluation,NaN,NaN,NaN,NaN
1,nn_cross_entropy,"ML-25M all 25,000,095 ratings",25000095,20000076,4fbcebdec690e07e15fbd5d5f0016a309ff5c2da8fb516aaae47c896568395ce,5,1024,"[0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0]",cross_entropy,False,"[(no genres listed), Action, Adventure, Animation, Children, Comedy, Crime, Documentary, Drama, Fantasy, Film-Noir, Horror, IMAX, Musical, Mystery, Romance, Sci-Fi, Thriller, War, Western]",11090510,1244.740440,False,0.789375,0.590068,0.402070,5000019,unchanged prior full fit plus original evaluation,NaN,NaN,NaN,NaN
2,nn_mse_with_genres,"ML-25M all 25,000,095 ratings",25000095,20000076,4fbcebdec690e07e15fbd5d5f0016a309ff5c2da8fb516aaae47c896568395ce,5,1024,"[0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0]",mse,True,"[(no genres listed), Action, Adventure, Animation, Children, Comedy, Crime, Documentary, Drama, Fantasy, Film-Noir, Horror, IMAX, Musical, Mystery, Romance, Sci-Fi, Thriller, War, Western]",11091601,654.316620,False,0.778726,0.584751,0.298490,5000019,recovery segment only; not uninterrupted five-epoch wall time,3.0,2.0,737.438,f417d7bc892df8e054b2ca789a2cb4606b8c296b7d5a4fa38a9ab2aacc287bbd


{
  "source": "https://codex.lenskit.org/movielens/ML25M/als-biased.html",
  "split_source": "https://codex.lenskit.org/movielens/ML25M/splitting.html",
  "reference_rmse": 0.7918119847363368,
  "reference_model": "LensKit ALS biased MF, 471 factors, tuned parameters",
  "reference_protocol": "global temporal train/tune/test windows; test users restricted to those seen in training",
  "our_protocol": "seed42 random row 80/20 validation split, no separate test/tuning window",
  "directly_comparable": false,
  "source_status": "work in progress; illustrative published online result, not a validated target",
  "differences_from_reference": {
    "nn_mse": -0.004316859358241332,
    "nn_cross_entropy": -0.0024374537789028095,
    "nn_mse_with_genres": -0.01308556849400766
  },
  "genre_improvement_rmse": 0.008768709135766328,
  "cross_entropy_minus_mse_rmse": 0.0018794055793385223
}
Cross-entropy predicts ten half-star classes on ML-25M; its probability-weighted expected rating allows an R

In [14]:
finish_run(COLAB_CONTEXT)

Saved run settings. Save the notebook in Colab to keep its cell outputs.


In [15]:
recovery.probe()


__HAILONG_LOCAL_MOVIELENS_EVIDENCE__{"backend": "local_native_windows_cuda", "in_colab": false, "drive_enabled": false, "quick_run": false, "all_original_fits": 11, "epochs_each": 5, "source_rows": 25000095, "full_fits": [{"model": "nn_mse", "observed_training_batches": 97660, "export_matches_final_fit": true, "validation_rows": 5000019}, {"model": "nn_cross_entropy", "observed_training_batches": 97660, "export_matches_final_fit": true, "validation_rows": 5000019}, {"model": "nn_mse_with_genres", "observed_training_batches": 97660, "export_matches_final_fit": true, "validation_rows": 5000019}], "source_sha256": "cf2cd8c9bd885d6df7c750705c2e17f9ab423cf56311362d5761b9b867f6cefc", "data_files": {"ml100k_ratings": {"path": "C:\\Users\\[LEARNER_NAME_OMITTED]\\.v\\hr\\20261019T063318-5946e910\\fastai-cache\\data\\ml-100k\\u.data", "bytes": 1979173, "sha256": "06416e597f82b7342361e41163890c81036900f418ad91315590814211dca490"}, "ml100k_movies": {"path": "C:\\Users\\[LEARNER_NAME_OMITTED]\\.v\\

{'backend': 'local_native_windows_cuda',
 'in_colab': False,
 'drive_enabled': False,
 'quick_run': False,
 'all_original_fits': 11,
 'epochs_each': 5,
 'source_rows': 25000095,
 'full_fits': [{'model': 'nn_mse',
   'observed_training_batches': 97660,
   'export_matches_final_fit': True,
   'validation_rows': 5000019},
  {'model': 'nn_cross_entropy',
   'observed_training_batches': 97660,
   'export_matches_final_fit': True,
   'validation_rows': 5000019},
  {'model': 'nn_mse_with_genres',
   'observed_training_batches': 97660,
   'export_matches_final_fit': True,
   'validation_rows': 5000019}],
 'source_sha256': 'cf2cd8c9bd885d6df7c750705c2e17f9ab423cf56311362d5761b9b867f6cefc',
 'data_files': {'ml100k_ratings': {'path': 'C:\\Users\\[LEARNER_NAME_OMITTED]\\.v\\hr\\20261019T063318-5946e910\\fastai-cache\\data\\ml-100k\\u.data',
   'bytes': 1979173,
   'sha256': '06416e597f82b7342361e41163890c81036900f418ad91315590814211dca490'},
  'ml100k_movies': {'path': 'C:\\Users\\[LEARNER_NAME_OM